# Korea Foreign Ownership - Research Record

Working record of the foreign-ownership flow research. Every number here was
computed by the pipeline in this repo from the exchange's end-of-day files;
nothing is hand-entered. Negative results are kept -- they are the answer,
not the absence of one.

| store | fields used | how the field is used |
| --- | --- | --- |
| foreign_ownership | foreign_pct, foreign_shares | the signal: level and daily change of foreign holding per issuer |
| foreign_ownership | shares_listed, foreign_limit_shares | ownership ratio denominators, statutory-limit usage |
| foreign_ownership | market | KOSPI / KOSDAQ split |
| market | close, value_traded | forward returns, ADV liquidity filter, KRW valuation of flows |
| market | market_cap | value-weighted market aggregates |

Coverage: ~3,600 issuers x ~4,100 trading sessions, collected daily on a
19:00 KST schedule and stored as immutable per-day parquet.

## Step 1 - Build the history

**Question: can a clean per-issuer daily history be built at all?**

Yes. The exchange publishes an end-of-day foreign-holding file per issuer;
the collector fetches missing days idempotently, so a skipped run never
becomes a gap. This dataset is the project's primary deliverable and every
later step consumes it.

| item | value |
| --- | --- |
| issuers | ~3,600 |
| sessions | ~4,100 |
| refresh | daily 19:00 KST, self-publishing |

In [ ]:
# Step 1 - collection pipeline (01_backfill.py, 05_daily_update.py)

## Step 2 - Persistence (the core hypothesis)

**Question: is foreign flow autocorrelated, as the brief guessed?**

Yes, strongly. Cross-sectional rank autocorrelation of daily flow at lag 1 is
+0.088 with t +64.7, and it survives a tie-artifact control (98% of the
effect retained). The premise of the whole idea is true.

| lag | rank AC |
| --- | --- |
| 1 | +0.088 (t +64.7) |
| tie-artifact control | 98% retained |

In [ ]:
# Step 2 - persistence (04_validate.py)

## Step 3 - Does it predict returns?

**Question: does today's foreign flow predict tomorrow's cross-section?**

Only nominally. The 1-day rank IC is +0.0039 (t +3.9) -- real but tiny --
and by 20 days the sign flips (-0.0039, Newey-West t -4.2). Whatever is
being anticipated at one day is given back within a month.

| horizon | rank IC | t |
| --- | --- | --- |
| 1d | +0.0039 | +3.9 |
| 20d | -0.0039 | -4.2 (NW) |

In [ ]:
# Step 3 - horizon IC (04_validate.py)

## Step 4 - Where does the effect live?

**Question: is the 1-day effect in names that can actually be traded?**

No. Split by liquidity, the effect concentrates in the illiquid tail. In the
liquid top third of the market -- the only place size could be deployed --
the IC is statistically zero.

| universe | 1d IC | t |
| --- | --- | --- |
| all names | +0.0039 | +3.9 |
| liquid top third | +0.0007 | +0.7 |

In [ ]:
# Step 4 - liquidity split (12_combination.py)

## Step 5 - Which t-statistics can be trusted?

**Question: daily ICs are autocorrelated (the signal itself persists) -- which
standard error is honest?**

A two-arm null calibration: feed each estimator data that is truly null under
(a) an MA(h) process and (b) white noise, and check its rejection rate. Naive
SEs over-reject under (a); a non-overlapping estimator under-rejects to 0%
under (b). Newey-West is the only one calibrated under both, so every
headline t in this record is NW. (The non-overlapping estimator was my first
choice; the calibration showed it was the worst of the three, and the
verdict was switched.)

In [ ]:
# Step 5 - SE calibration (09_overlap_correction.py, 09b_se_diagnostic.py)

## Step 6 - Level signals

**Question: does the LEVEL of foreign ownership (not its change) carry alpha?**

It looks like it until autocorrelation is priced in. The 60d level IC has a
naive t of +35.9, but the daily IC series has AC(1) = 0.974: Newey-West cuts
the t to +5.95 and the effective sample is 43 observations out of 1,567.
That is a handful of regimes, not an edge -- classified as regime exposure
under the rule set before the test was run.

| signal | naive t | NW t | n_eff |
| --- | --- | --- | --- |
| level 60d | +35.9 | +5.95 | 43 / 1,567 |

In [ ]:
# Step 6 - level signals (11_level_se.py)

## Step 7 - Costs

**Question: does anything survive real Korean trading costs?**

No. The 1-day signal needs ~320 rebalances a year at 63.5% turnover each;
at 30bp round trip plus the 0.20% securities transaction tax the long-short
book loses -89.7% a year. The signal decays faster than it can be traded.

| item | value |
| --- | --- |
| turnover per rebalance | 63.5% |
| rebalances / yr (1d) | ~320 |
| net at 30bp + tax | -89.7% / yr |

In [ ]:
# Step 7 - cost model (04_validate.py section 2c)

## Step 8 - The comparison nobody asked for

**Question: if persistence is the reason to follow a flow, is FOREIGN flow
the most persistent one?**

No -- domestic institutions are. Measured on the same basis, institutional
flow autocorrelation is ~2.3x the foreign figure at every lag out to 20.
The brief's mechanism ("foreign investors digest information slowly") is
refuted in its comparative form; the persistence itself is real but foreign
flow is not where it is strongest.

| series | lag-1 AC | t |
| --- | --- | --- |
| domestic institutions | +0.298 | +235.9 |
| foreign | +0.130 | +95.2 |

In [ ]:
# Step 8 - investor comparison (04_validate.py section 5)

## Step 9 - Index-rebalance denoising

**Question: can MSCI/FTSE rebalance noise be filtered out, as planned?**

Not with a calendar. Measured flow on 136 known effective dates is 1.02x a
normal day -- indistinguishable. Either the flow spreads across surrounding
days or the store's holding series absorbs it differently; either way there
is no event spike to filter, so this brief item is blocked, not skipped.

| test | result |
| --- | --- |
| effective-date flow multiple | 1.02x (136 dates) |

In [ ]:
# Step 9 - rebalance calendar test (krxflow/rebalance.py)

## Step 10 - Orthogonality and combination

**Question: the brief's real intent -- is this useful ALONGSIDE other features?**

Half-answered. The signal survives residualisation against size, liquidity,
reversal, momentum and volatility, so its information is not already in the
standard set -- the property that matters for combination. But the factor
composite built here as a stand-in baseline is itself unprofitable
(IC -0.078), so a marginal-contribution number on top of it would be
meaningless. The honest statement: combination cannot be judged without the
real production feature stack. That test is the natural next step and needs
access, not more data.

In [ ]:
# Step 10 - orthogonality (12_combination.py)

## Step 11 - The live tracker

**Question: what would riding the flow actually have earned? (recomputed on
every refresh)**

Buy the top decile of 20-session accumulation, hold h sessions, versus the
equal-weighted universe, 0.50% per round trip. Gross is statistically zero
at every horizon and at best a quarter of the cost hurdle; net is negative
everywhere. This table lives in docs/monitor.html and re-runs itself, so
the verdict stays current instead of being a one-time backtest.

| hold | trades | gross bp | net bp | t (NW) | hit | net /yr |
| --- | --- | --- | --- | --- | --- | --- |
| 1d | 619 | +0.7 | -49.3 | 0.30 | 51% | -123% |
| 5d | 123 | +4.2 | -45.8 | 0.34 | 55% | -23% |
| 20d | 30 | +12.8 | -37.2 | 0.28 | 57% | -4.6% |

In [ ]:
# Step 11 - forward tracker (tracker.py, monitor section)

## Step 12 - Open data puzzle

**Question: does the holdings-implied flow match the exchange's own reported
net buying?**

Only at +0.10 correlation across ~4,100 days x ~3,600 issuers. Candidate
explanations: settlement vs trade date, securities lending, custody
transfers, DR conversions, in-kind index creation. It does not change any
verdict above, but it changes what the series measures. Next data step:
collect the official investor-type net-buying series (free) alongside.

In [ ]:
# Step 12 - holdings vs reported flow cross-check

## Every design decision, side by side

| decision | alternatives tested | chosen | why |
| --- | --- | --- | --- |
| standard error | naive / non-overlapping / Newey-West | Newey-West | only one calibrated under both nulls |
| signal form | daily change / level / 60-120d change | none deployable | level = regime exposure; changes die at cost |
| universe | all / liquid top third | liquid third for any trading claim | effect elsewhere is untradable |
| cost model | zero / 30bp / 30bp + 0.20% tax | 30bp + tax | Korean sell-side tax is unavoidable |
| forward returns | raw / split-guarded | split-guarded | unadjusted prices fake -90% days |

## Bottom line

- **Dataset (deliverable): done and self-maintaining.** Daily per-issuer
  foreign ownership, published to the repo on a daily schedule.
- **Hypothesis: premise confirmed** (persistence is real, t +64.7), but the
  comparative story is inverted -- institutions are 2.3x more persistent.
- **Standalone trading: refuted three ways** (untradable universe, 20d sign
  flip, costs), and a live tracker keeps re-checking it.
- **Combination with production features: the remaining question**, and the
  one the dataset was built for. Needs feature-stack access, not more data.